In [1]:
# pip → Python package manager
# Used to install Python libraries from PyPI (Python Package Index)

# install → command used to install libraries/packages

# -U means "Upgrade"
# If library already exists, pip upgrades it to latest version

# transformers → Hugging Face library for loading and using transformer models
# Used for:
# - LLMs (GPT, Llama, Qwen, Mistral)
# - Tokenizers
# - Training
# - Inference 
# - Text generation

# peft → Parameter Efficient Fine-Tuning library
# Used for efficient fine-tuning methods like:
# - LoRA
# - QLoRA
# - Prefix tuning
# Instead of training full model, it trains small adapter layers

# trl → Transformer Reinforcement Learning library
# Used for:
# - SFT (Supervised Fine-Tuning)
# - RLHF
# - DPO
# - Chatbot alignment
# Provides trainers specially designed for LLM fine-tuning

# accelerate → Hugging Face library for optimized GPU/distributed training
# Handles:
# - Multi-GPU training
# - Mixed precision (FP16/BF16)
# - Device placement
# - Memory optimization

# bitsandbytes → Quantization and memory optimization library
# Used for:
# - 8-bit loading
# - 4-bit loading
# - QLoRA
# - 8-bit optimizers
# Reduces GPU memory usage dramatically

# datasets → Hugging Face dataset handling library
# Used to:
# - Load datasets
# - Process data
# - Shuffle/batch/tokenize data
# - Create training datasets efficiently

!pip install -U transformers peft trl accelerate bitsandbytes datasets
!pip install --upgrade torchao

In [4]:
from datasets import load_dataset

dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})

In [8]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})


In [13]:
# Define a function named format_prompt
# This function takes one dataset example (one row/sample) as input

def format_prompt(example):

    # f""" """ → Python multiline f-string
    # Used to create formatted text dynamically

    # We are creating the FINAL TRAINING PROMPT here
    # This converts raw dataset columns into instruction-tuning format

    text = f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""

    # example['instruction']
    # Accesses the "instruction" column from dataset

    # example['input']
    # Accesses additional context/input column

    # example['output']
    # Accesses expected answer/target response

    # Final formatted text looks like:

    # ### Instruction:
    # Explain AI

    # ### Input:
    # Beginner friendly

    # ### Response:
    # AI is the simulation of human intelligence...



    # Return a dictionary
    # Dataset.map() expects returned output in dictionary format

    return {

        # Create new column named "text"
        # This column will contain final formatted prompt

        "text": text
    }


# dataset.map()
# Applies function to EVERY row/sample in dataset

# For each row:
# 1. format_prompt() runs
# 2. formatted text is created
# 3. new "text" column added

dataset = dataset.map(format_prompt)


Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

In [12]:
dataset['train']

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 112165
    })
})

In [ ]:
print(dataset["train"].column_names)

In [14]:
print(dataset["train"][0]["text"])


### Instruction:
If you are a doctor, please answer the medical questions based on the patient's description.

### Input:
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!

### Response:
Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nau

In [22]:
small_dataset = dataset["train"].select(range(1000))

In [23]:
# Import AutoTokenizer class from transformers library
# AutoTokenizer automatically loads the correct tokenizer
# for the selected transformer model

from transformers import AutoTokenizer


# Import AutoModelForCausalLM class from transformers library
# Used to load GPT-style causal language models
# such as:
# - GPT
# - Llama
# - Qwen
# - Mistral
# - Gemma

from transformers import AutoModelForCausalLM


# print() displays output on console/screen
# Used here just to verify that imports worked successfully

print("Imports successful")

Imports successful


In [24]:
# Import PyTorch library
# PyTorch is the deep learning framework used for:
# - tensors
# - GPU computation
# - neural networks
# - transformer execution
# - backpropagation

import torch


# Store model name inside variable
# "Qwen/Qwen2.5-1.5B-Instruct" is the Hugging Face model repository name

# Qwen → model family
# 2.5 → model version
# 1.5B → approximately 1.5 billion parameters
# Instruct → instruction fine-tuned/chat version

model_name = "Qwen/Qwen2.5-1.5B-Instruct"


# Load tokenizer for this model

# AutoTokenizer automatically:
# - downloads tokenizer files
# - loads vocabulary
# - loads special tokens
# - loads tokenization rules

# Tokenizer converts:
# Human text → token IDs

tokenizer = AutoTokenizer.from_pretrained(model_name)


# Load the actual transformer model

# AutoModelForCausalLM loads:
# - transformer architecture
# - pretrained weights
# - language modeling head

# from_pretrained() downloads pretrained model weights
# from Hugging Face Hub if not already cached locally

model = AutoModelForCausalLM.from_pretrained(

    # model repository name
    model_name,


    # torch_dtype=torch.float16
    # Load model weights in FP16 (16-bit floating point)

    # Benefits:
    # - lower GPU memory usage
    # - faster computation
    # - optimized tensor core execution

    # FP16 uses half memory compared to FP32

    torch_dtype=torch.float16,


    # device_map="auto"
    # Automatically decides where model should be loaded

    # Usually:
    # - GPU if available
    # - CPU otherwise

    # For large models:
    # Hugging Face can automatically split layers
    # across multiple GPUs

    device_map="auto"
)


# Print success message
# Helps verify model loaded correctly

print("Model Loaded Successfully")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model Loaded Successfully


In [25]:
# Import LoRA configuration class from PEFT library
# LoraConfig is used to define:
# - LoRA rank
# - target layers
# - dropout
# - scaling factors

# Import get_peft_model function
# This function injects LoRA adapters into transformer model

from peft import (
    LoraConfig,
    get_peft_model
)



# Create LoRA configuration object

lora_config = LoraConfig(

    # r = LoRA rank
    # Controls size/capacity of trainable adapter matrices

    # Smaller r:
    # - lower memory
    # - faster training

    # Larger r:
    # - more learning capacity
    # - more trainable parameters

    r=8,


    # lora_alpha = scaling factor
    # Controls strength of LoRA updates

    # Internally:
    # effective update = (alpha/r) * BA

    # Higher alpha:
    # stronger LoRA influence

    lora_alpha=16,


    # target_modules defines WHERE LoRA adapters
    # should be inserted inside transformer

    # q_proj = Query projection layer
    # v_proj = Value projection layer

    # These are attention layers inside transformer

    # LoRA adapters are attached only to these layers

    target_modules=[
        "q_proj",
        "v_proj"
    ],


    # Dropout applied to LoRA layers during training

    # Helps prevent overfitting

    # 0.05 means:
    # 5% activations randomly dropped during training

    lora_dropout=0.05,


    # bias controls whether bias parameters are trainable

    # "none" means:
    # do NOT train bias terms

    # Saves memory and parameters

    bias="none",


    # task_type specifies type of NLP task

    # CAUSAL_LM means:
    # GPT-style next-token prediction

    # Used for:
    # - Qwen
    # - GPT
    # - Llama
    # - Mistral

    task_type="CAUSAL_LM"
)



# Inject LoRA adapters into model

# model = original transformer model
# lora_config = LoRA settings

# This converts:
# Normal model → PEFT-enabled LoRA model

model = get_peft_model(
    model,
    lora_config
)



# Print trainable parameter statistics

# Shows:
# - trainable params
# - total params
# - trainable percentage

# Useful for verifying LoRA works correctly

model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [26]:
# Import TrainingArguments class from transformers library
# TrainingArguments is used to configure the entire training process

from transformers import TrainingArguments



# Create training configuration object

training_args = TrainingArguments(

    # output_dir specifies where training outputs/checkpoints
    # will be saved

    # Examples of saved files:
    # - model checkpoints
    # - optimizer states
    # - logs
    # - trainer state

    output_dir="./results",

    # per_device_train_batch_size means:
    # number of training samples processed at one time
    # PER GPU/device

    # batch size = 2 means:
    # 2 samples processed in one forward/backward pass

    # Small batch sizes are common for LLMs because:
    # - models are huge
    # - GPU memory is limited

    per_device_train_batch_size=2,



    # gradient_accumulation_steps allows effective larger batch size
    # without increasing GPU memory usage

    # Here:
    # gradients from 2 steps are accumulated
    # before optimizer updates weights

    # Effective batch size formula:
    # batch_size × accumulation_steps

    # Effective batch size here:
    # 2 × 2 = 4

    gradient_accumulation_steps=2,



    # learning_rate controls how much model weights update
    # during training

    # 2e-4 means:
    # 0.0002

    # Smaller learning rate:
    # - stable learning
    # - slower training

    # Larger learning rate:
    # - faster learning
    # - unstable training risk

    learning_rate=2e-4,



    # num_train_epochs means:
    # how many complete passes over dataset

    # 3 epochs means:
    # model sees entire dataset 3 times

    num_train_epochs=3,



    # logging_steps controls how often training logs print

    # logging_steps=1 means:
    # print logs every training step

    # Example:
    # step=1 loss=1.82

    logging_steps=1,



    # save_steps controls checkpoint saving frequency

    # save_steps=50 means:
    # save model checkpoint every 50 steps

    # Useful for:
    # - recovery after crashes
    # - resuming training
    # - evaluation

    save_steps=50,



    # fp16=True enables FP16 mixed precision training

    # FP16 = 16-bit floating point precision

    # Benefits:
    # - lower GPU memory
    # - faster training
    # - better tensor core utilization

    fp16=True,



    # optim specifies optimizer type

    # paged_adamw_8bit:
    # - memory optimized AdamW optimizer
    # - uses 8-bit optimizer states
    # - provided by bitsandbytes library

    # "paged":
    # prevents GPU memory spikes

    # "8bit":
    # reduces optimizer memory usage

    optim="paged_adamw_8bit"
)

In [27]:
# Import SFTTrainer from TRL library

# SFTTrainer = Supervised Fine-Tuning Trainer

# Specialized trainer designed for:
# - instruction tuning
# - chatbot fine-tuning
# - conversational LLM training

# It simplifies:
# - tokenization
# - batching
# - loss calculation
# - training loop
# - padding
# - causal LM handling

from trl import SFTTrainer



# Create trainer object

trainer = SFTTrainer(

    # model = transformer model to fine-tune

    # Usually this is:
    # - quantized model
    # - LoRA-enabled model
    # - causal language model

    # This model already contains:
    # - pretrained knowledge
    # - LoRA adapters

    model=model,



    # train_dataset = dataset used for training

    # Contains formatted instruction-response examples

    # Example:
    # {
    #   "text": "### Instruction: ... "
    # }

    train_dataset=small_dataset,



    # processing_class = tokenizer

    # Tokenizer converts:
    # text → token IDs

    # It also handles:
    # - padding
    # - truncation
    # - attention masks

    # New TRL versions use:
    # processing_class
    # instead of:
    # tokenizer

    processing_class=tokenizer,



    # args = TrainingArguments object

    # Controls:
    # - batch size
    # - epochs
    # - learning rate
    # - fp16
    # - optimizer
    # - logging

    args=training_args,



    # formatting_func defines how to extract final text
    # from dataset rows

    # lambda x: x["text"]
    # means:
    # take "text" column from dataset

    # x represents one dataset row/sample

    formatting_func=lambda x: x["text"]
)

Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [28]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,2.875621
2,2.738643
3,2.809134
4,2.919301
5,2.817412
6,2.594800
7,2.927776
8,2.754650
9,2.604702
10,2.787703


TrainOutput(global_step=750, training_loss=2.3220498345692953, metrics={'train_runtime': 413.2593, 'train_samples_per_second': 7.259, 'train_steps_per_second': 1.815, 'total_flos': 6929339840587776.0, 'train_loss': 2.3220498345692953})

In [ ]:
trainer.save_model("./fine_tuned_model")

print("Fine-tuned model saved")

In [ ]:
# Create input prompt for the model

# This is the user query/question
# that will be given to the language model

prompt = "I have headache since morning and I'm feeling tired. Suggest me a medicine"



# Tokenize the prompt

# tokenizer() converts:
# text → token IDs

# return_tensors="pt"
# tells tokenizer to return PyTorch tensors

# .to(model.device)
# moves tensors to same device as model
# usually GPU

inputs = tokenizer(

    # input text
    prompt,


    # return tensors in PyTorch format

    return_tensors="pt"

).to(model.device)



# Generate response using model

outputs = model.generate(

    # **inputs means:
    # unpack dictionary values as keyword arguments

    # Equivalent to:
    # input_ids=inputs["input_ids"]
    # attention_mask=inputs["attention_mask"]

    **inputs,


    # max_new_tokens controls:
    # maximum number of tokens model can generate

    # Here:
    # model can generate up to 100 new tokens

    max_new_tokens=100
)



# Convert generated token IDs back into human-readable text

response = tokenizer.decode(

    # outputs[0]
    # select first generated sequence

    outputs[0],


    # skip_special_tokens removes special tokens like:
    # <pad>
    # <eos>
    # <bos>

    skip_special_tokens=True
)



# Print final generated response

print(response)